In [ ]:
%load_ext autoreload
%autoreload 2


In [1]:
import os
import shutil
from glob import glob

import numpy as np
from skl2onnx.helpers.onnx_helper import load_onnx_model, save_onnx_model
from tqdm import tqdm

from service.fragment.create_fragments import get_fragments
from service.fragment.net import Net
from utilities.change_model_layers_dimension import change_layers_dimension


2025-11-23 14:42:03.423825662 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


In [2]:
x = np.random.randn(1, 3, 224, 224).astype(np.float32)

nets = []
# Use fine-tuned models if available, otherwise use pre-trained models
model_paths = sorted(glob("../_results_chest_xray/finetune/*/model_ft.onnx"))
if len(model_paths) == 0:
    # Fall back to pre-trained models
    model_paths = sorted(glob("../_models_chest_xray/*.onnx"))

for i, model in tqdm(enumerate(model_paths), position=0, leave=True):
    print(model)
    if "auto" in model or "rnn" in model:
        continue
    model_onnx1 = load_onnx_model(model)
    change_layers_dimension(model_onnx1)
    fragments1 = get_fragments(model_onnx1, x)
    net1 = Net(fragments1, i)
    nets.append(net1)


0it [00:00, ?it/s]

../_results_chest_xray/finetune/alexnet/model_ft.onnx


2025-11-23 14:42:15.281178897 [W:onnxruntime:, transformer_memcpy.cc:111 ApplyImpl] 2 Memcpy nodes are added to the graph main_graph for CUDAExecutionProvider. It might have negative impact on performance (including unable to run CUDA graph). Set session_options.log_severity_level=1 to see the detail logs before this message.
1it [00:14, 14.16s/it]

../_results_chest_xray/finetune/densenet121/model_ft.onnx


2025-11-23 14:42:21.415577301 [W:onnxruntime:, transformer_memcpy.cc:111 ApplyImpl] 2 Memcpy nodes are added to the graph main_graph for CUDAExecutionProvider. It might have negative impact on performance (including unable to run CUDA graph). Set session_options.log_severity_level=1 to see the detail logs before this message.
2025-11-23 14:42:21.517311717 [W:onnxruntime:, transformer_memcpy.cc:111 ApplyImpl] 2 Memcpy nodes are added to the graph main_graph for CUDAExecutionProvider. It might have negative impact on performance (including unable to run CUDA graph). Set session_options.log_severity_level=1 to see the detail logs before this message.
2025-11-23 14:42:21.650734762 [W:onnxruntime:, transformer_memcpy.cc:111 ApplyImpl] 2 Memcpy nodes are added to the graph main_graph for CUDAExecutionProvider. It might have negative impact on performance (including unable to run CUDA graph). Set session_options.log_severity_level=1 to see the detail logs before this message.
2025-11-23 14:42

...

3it [00:28,  7.83s/it]

../_results_chest_xray/finetune/resnet50/model_ft.onnx


2025-11-23 14:42:34.876012878 [E:onnxruntime:Default, cuda_call.cc:123 CudaCall] CUDNN failure 2000: CUDNN_STATUS_BAD_PARAM ; GPU=0 ; hostname=LIA-WS-045 ; file=/onnxruntime_src/onnxruntime/core/providers/cuda/nn/conv.cc ; line=492 ; expr=cudnnAddTensor(cudnn_handle, &alpha, s_.b_tensor, s_.b_data, &alpha, s_.y_tensor, s_.y_data); 
2025-11-23 14:42:34.876028961 [E:onnxruntime:, sequential_executor.cc:572 ExecuteKernel] Non-zero status code returned while running Conv node. Name:'node_Conv_649' Status Message: CUDNN failure 2000: CUDNN_STATUS_BAD_PARAM ; GPU=0 ; hostname=LIA-WS-045 ; file=/onnxruntime_src/onnxruntime/core/providers/cuda/nn/conv.cc ; line=492 ; expr=cudnnAddTensor(cudnn_handle, &alpha, s_.b_tensor, s_.b_data, &alpha, s_.y_tensor, s_.y_data); 
2025-11-23 14:42:35.450769198 [E:onnxruntime:Default, cuda_call.cc:123 CudaCall] CUDNN failure 2000: CUDNN_STATUS_BAD_PARAM ; GPU=0 ; hostname=LIA-WS-045 ; file=/onnxruntime_src/onnxruntime/core/providers/cuda/nn/conv.cc ; line=492 

../_results_chest_xray/finetune/vgg16/model_ft.onnx


2025-11-23 14:43:53.858295237 [W:onnxruntime:, transformer_memcpy.cc:111 ApplyImpl] 2 Memcpy nodes are added to the graph main_graph for CUDAExecutionProvider. It might have negative impact on performance (including unable to run CUDA graph). Set session_options.log_severity_level=1 to see the detail logs before this message.
5it [01:55, 23.18s/it]


In [3]:
for net in nets:
    print("number of fragments", len(net))


number of fragments 8
number of fragments 5
number of fragments 13
number of fragments 6
number of fragments 16


In [4]:
shutil.rmtree("../_results_chest_xray/fragments", ignore_errors=True)
os.makedirs("../_results_chest_xray/fragments", exist_ok=True)
for i, net in enumerate(nets):
    for j, fragment in enumerate(net):
        folder = f"../_results_chest_xray/fragments/net{i:03}/"
        os.makedirs(folder, exist_ok=True)
        filename = folder + f"fragment{j:03}.onnx"
        save_onnx_model(fragment.fragment, filename)
